# Gamma UMAP maps — CPU implementation

- Gammas: **0, 0.25, 0.5, 0.75, 1**; PCA: 50 components; graph: 15 non-self neighbors; local scale: 10th neighbor; candidate pool: 100.
- Formula from `gamma_scaling.py`: `D_gamma(i,j) = distance(i,j)**2 / (sigma_i * sigma_j)**gamma`. Neighbors are reranked within the candidate pool, as in the original helper.
- The original embedding helper's preprocessing is preserved: use the `counts` layer if present, otherwise `.X`; normalize totals to 10,000, log1p, then PCA once per dataset. Your revised `run_sweep.prepare` uses `.X` directly, so these agree when `.X` contains the same counts.
- Each UMAP receives its gamma-specific neighbor indices **and scaled distance values** through `precomputed_knn`; it does not rebuild an ordinary Euclidean graph. See the [UMAP API documentation](https://umap-learn.readthedocs.io/en/latest/api.html).
- CPU UMAP coordinates can differ from cuML's. Outputs are saved separately in `results/gamma_umap_cpu/` so old GPU results are not reused.

Edit `DRIVE_ROOT` in the paths cell if necessary. The `*_gpu.h5ad` suffix is just the input filename; these files can be read on CPU. Set `LIMIT = 1` for a first dataset, then `None` for all datasets.


In [1]:
# Run before importing scientific packages.
import subprocess
import sys
from importlib import metadata

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "scanpy", "umap-learn"],
    check=True,
)
import numpy as np
import scanpy as sc
import anndata
from sklearn.neighbors import NearestNeighbors
from umap import UMAP

for package in ("numpy", "scanpy", "anndata", "scikit-learn", "umap-learn"):
    print(package, metadata.version(package))
print("CPU imports passed.")


/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


numpy 2.1.3
scanpy 1.12.5
anndata 0.13.4
scikit-learn 1.6.1
umap-learn 0.5.12
CPU imports passed.


In [6]:
from pathlib import Path

try:
    from google.colab import drive
except ImportError:
    DRIVE_ROOT = Path.cwd()
else:
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/26F/CS6850/CS 5850: Exploratory Paper Project/Notebooks/")

DATA_DIR = DRIVE_ROOT / "data"
RESULT_DIR = DRIVE_ROOT / "results/gamma_umap_cpu"
LIMIT = 1  # Use 1 for a first check, None for every available dataset.
OVERWRITE = False

files = sorted(DATA_DIR.rglob("*sce*_gpu.h5ad"))
if not files:
    raise FileNotFoundError(f"No converted datasets under {DATA_DIR}. Edit DRIVE_ROOT or DATA_DIR above.")
RESULT_DIR.mkdir(parents=True, exist_ok=True)
print("Datasets found:", len(files))
for path in files:
    print(" ", path.name)
print("Results:", RESULT_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Datasets found: 12
  sce_filteredHVG10_KohTCC_gpu.h5ad
  sce_filteredHVG10_Koh_gpu.h5ad
  sce_filteredHVG10_KumarTCC_gpu.h5ad
  sce_filteredHVG10_Kumar_gpu.h5ad
  sce_filteredHVG10_SimKumar4easy_gpu.h5ad
  sce_filteredHVG10_SimKumar4hard_gpu.h5ad
  sce_filteredHVG10_SimKumar8hard_gpu.h5ad
  sce_filteredHVG10_TrapnellTCC_gpu.h5ad
  sce_filteredHVG10_Trapnell_gpu.h5ad
  sce_filteredHVG10_Zhengmix4eq_gpu.h5ad
  sce_filteredHVG10_Zhengmix4uneq_gpu.h5ad
  sce_filteredHVG10_Zhengmix8eq_gpu.h5ad
Results: /content/drive/MyDrive/26F/CS6850/CS 5850: Exploratory Paper Project/Notebooks/results/gamma_umap_cpu


## CPU embedding functions

Self-contained replacements for the GPU helper. PCA and candidate neighbors are computed once per dataset and reused across gamma values.


In [7]:
import gc
import os
from pathlib import Path

import numpy as np

# CPU gamma scaling; formula copied from gamma_scaling.py.
def gamma_scaled(dist, ind, gamma, k_sigma=10, eps=1e-12):
    if not 1 <= k_sigma <= dist.shape[1]:
        raise ValueError("k_sigma must fit within the candidate pool")
    if gamma == 0:
        return dist ** 2
    sigma = np.maximum(dist[:, k_sigma - 1], eps)
    return dist ** 2 / (sigma[:, None] * sigma[ind]) ** gamma


GAMMAS = (0.0, 0.25, 0.5, 0.75, 1.0)


def prepare_cpu(adata, n_comps=50, seed=0):
    """Keep the original embedding helper's counts-layer choice and preprocessing."""
    import scanpy as sc
    if "counts" in adata.layers:
        adata.X = adata.layers["counts"].copy()
    for layer in list(adata.layers):
        del adata.layers[layer]
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    n_pc = min(n_comps, adata.n_vars - 1, adata.n_obs - 1)
    if n_pc < 1:
        raise ValueError("PCA needs at least two cells and genes")
    sc.pp.pca(adata, n_comps=n_pc, mask_var=None, svd_solver="arpack", random_state=seed)
    return np.asarray(adata.obsm["X_pca"], dtype=np.float64)


def candidate_neighbors(X, candidate_k=100, metric="euclidean"):
    """Exact self-free neighbours on CPU. Independent of gamma."""
    from sklearn.neighbors import NearestNeighbors

    n = X.shape[0]
    pool = min(candidate_k, n - 1)
    model = NearestNeighbors(
        n_neighbors=pool + 1, algorithm="brute", metric=metric)
    model.fit(X)
    dist, ind = model.kneighbors(X)
    dist = np.asarray(dist)
    ind = np.asarray(ind).astype(np.int32)
    rows = np.arange(n, dtype=np.int32)[:, None]
    keep = np.argsort(np.where(ind == rows, np.inf, dist), axis=1)[:, :pool]
    dist = np.take_along_axis(dist, keep, axis=1)
    ind = np.take_along_axis(ind, keep, axis=1)
    if bool(np.any(ind == rows)):
        raise RuntimeError("self neighbour survived the filter")
    return dist, ind


def gamma_graph(dist, ind, k, gamma, k_sigma):
    """Top-k neighbours under D_gamma, still without the self column."""

    reduced = gamma_scaled(dist, ind, gamma, k_sigma)
    order = np.argsort(reduced, axis=1)[:, :k]
    chosen = np.take_along_axis(ind, order, axis=1)
    values = np.take_along_axis(reduced, order, axis=1)
    return chosen, values


def embed_umap(X, ind, dist, seed=0):
    """Use the gamma graph directly, including self in column zero.

    force_approximation_algorithm ensures UMAP honors precomputed_knn even
    for fewer than 4096 cells. A search index is unnecessary for fit_transform.
    """
    from umap import UMAP
    n = ind.shape[0]
    knn_ind = np.ascontiguousarray(np.column_stack([
        np.arange(n, dtype=np.int32), ind.astype(np.int32)]))
    knn_dist = np.ascontiguousarray(np.column_stack([
        np.zeros(n, dtype=np.float32), dist.astype(np.float32)]))
    reducer = UMAP(
        n_neighbors=knn_ind.shape[1], n_components=2, min_dist=0.1,
        metric="euclidean", random_state=seed, n_jobs=1,
        precomputed_knn=(knn_ind, knn_dist, None),
        force_approximation_algorithm=True,
    )
    return np.asarray(reducer.fit_transform(X), dtype=np.float32)


def output_path(out_dir, stem, n_comps, k_graph, k_sigma, seed, gamma):
    name = (
        f"{stem}_pca{n_comps}_k{k_graph}_ksigma{k_sigma}"
        f"_seed{seed}_gamma{gamma:.2f}_umap.npz"
    )
    return Path(out_dir) / name


def embed_dataset(path, out_dir, gammas=GAMMAS, n_comps=50, k_graph=15,
                  k_sigma=10, candidate_k=100, seed=0, overwrite=False):
    """Write one embedding per gamma for a converted *_gpu.h5ad file."""
    import anndata

    path = Path(path)
    stem = path.name.removesuffix("_gpu.h5ad").removesuffix(".h5ad")
    pending = [
        g for g in gammas
        if overwrite or not output_path(
            out_dir, stem, n_comps, k_graph, k_sigma, seed, g).exists()
    ]
    if not pending:
        print(f"skip {path.name}: all gammas already saved", flush=True)
        return []

    Path(out_dir).mkdir(parents=True, exist_ok=True)
    adata = anndata.read_h5ad(path)
    if "phenoid" not in adata.obs:
        raise KeyError(f"{path.name}: missing phenoid cell-type labels")
    phenoid = np.asarray(adata.obs["phenoid"].astype(str), dtype=str)
    obs_names = np.asarray(adata.obs_names.astype(str), dtype=str)
    print(f"{path.name}: {adata.n_obs} cells, {adata.n_vars} genes", flush=True)
    X = prepare_cpu(adata, n_comps=n_comps, seed=seed)
    if X.shape[0] <= k_graph + 1:
        raise ValueError(f"{path.name} has too few cells for k_graph={k_graph}")

    pool = max(k_graph, candidate_k, k_sigma)
    dist, ind = candidate_neighbors(X, candidate_k=pool)
    if k_sigma > dist.shape[1]:
        raise ValueError("Not enough cells to estimate the requested local scale")
    written = []
    for gamma in pending:
        print(f"  gamma={gamma:.2f}", flush=True)
        chosen, values = gamma_graph(dist, ind, k_graph, gamma, k_sigma)
        embedding = embed_umap(X, chosen, values, seed=seed)
        if embedding.shape != (X.shape[0], 2):
            raise RuntimeError(
                f"expected {(X.shape[0], 2)} embedding, got {embedding.shape}")
        destination = output_path(
            out_dir, stem, n_comps, k_graph, k_sigma, seed, gamma)
        temporary = destination.with_suffix(".tmp.npz")
        payload = dict(embedding=embedding, obs_names=obs_names, gamma=np.float64(gamma))
        if phenoid is not None:
            payload["phenoid"] = phenoid
        np.savez_compressed(temporary, **payload)
        os.replace(temporary, destination)
        written.append(destination)
        del chosen, values, embedding
    del adata, X, dist, ind
    gc.collect()
    return written


def run(data_dir, out_dir, gammas=GAMMAS, n_comps=50, k_graph=15, k_sigma=10,
        candidate_k=100, seed=0, limit=None, overwrite=False):
    """Embed every converted HVG10 dataset in data_dir."""
    data_dir = Path(data_dir)
    out_dir = Path(out_dir)
    files = sorted(data_dir.rglob("*sce*_gpu.h5ad"))
    if not files:
        raise FileNotFoundError(
            f"No *_gpu.h5ad files in {data_dir}. Run convert_rds_gpu.py first.")
    if limit is not None:
        files = files[:limit]
    out_dir.mkdir(parents=True, exist_ok=True)
    written = []
    for path in files:
        written.extend(embed_dataset(
            path, out_dir, gammas=gammas, n_comps=n_comps, k_graph=k_graph,
            k_sigma=k_sigma, candidate_k=candidate_k, seed=seed,
            overwrite=overwrite))
    print(f"Wrote {len(written)} embeddings to {out_dir}", flush=True)
    return written


In [8]:
written = run(
    DATA_DIR, RESULT_DIR,
    gammas=GAMMAS, n_comps=50, k_graph=15, k_sigma=10,
    candidate_k=100, seed=0, limit=LIMIT, overwrite=OVERWRITE,
)
print("New embeddings:", len(written))
print("Saved embeddings:", len(list(RESULT_DIR.glob("*_umap.npz"))))


sce_filteredHVG10_KohTCC_gpu.h5ad: 531 cells, 81194 genes


AttributeError: 'NoneType' object has no attribute 'dtype'

# Point maps

One figure per dataset. Each panel is the 2D embedding for one gamma, and color is the `phenoid` cell-type label.

In [ ]:
import re
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np
from matplotlib import colormaps

pattern = re.compile(r'^(?P<group>.+)_gamma(?P<gamma>[0-9.]+)_umap\.npz$')
groups = defaultdict(list)
for path in sorted(RESULT_DIR.glob('*_umap.npz')):
    match = pattern.match(path.name)
    if match:
        groups[match.group('group')].append((float(match.group('gamma')), path))
if not groups:
    raise FileNotFoundError(f'No *_umap.npz files in {RESULT_DIR}')

figure_dir = RESULT_DIR / 'figures'
figure_dir.mkdir(parents=True, exist_ok=True)

for group, items in groups.items():
    items.sort()
    loaded = [(gamma, np.load(path, allow_pickle=False)) for gamma, path in items]
    labels = loaded[0][1]['phenoid'].astype(str)
    categories = sorted(set(labels.tolist()))
    palette = colormaps['tab20'].resampled(max(len(categories), 1))
    color_of = {label: palette(i) for i, label in enumerate(categories)}
    point_size = float(np.clip(8000 / len(labels), 4, 18))

    fig, axes = plt.subplots(
        1, len(loaded), figsize=(3.5 * len(loaded), 4.6), squeeze=False)
    for ax, (gamma, stored) in zip(axes[0], loaded):
        xy = stored['embedding']
        panel_labels = stored['phenoid'].astype(str)
        for label in categories:
            mask = panel_labels == label
            ax.scatter(
                xy[mask, 0], xy[mask, 1], s=point_size, c=[color_of[label]],
                linewidths=0, alpha=0.85, rasterized=True)
        ax.set_title(f'gamma = {gamma:.2f}')
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_aspect('equal', adjustable='datalim')
        stored.close()

    short = group.split('_pca')[0].removeprefix('sce_filteredHVG10_')
    fig.suptitle(short)
    fig.tight_layout()
    fig.legend(
        [plt.Line2D([0], [0], marker='o', color='none', markerfacecolor=color_of[label], markersize=7)
         for label in categories],
        categories, loc='lower center', ncol=min(len(categories), 6),
        bbox_to_anchor=(0.5, -0.02), frameon=False)
    figure_path = figure_dir / f'{group}_maps.png'
    fig.savefig(figure_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(figure_path)